# 00 - ETL: Data Consolidation and Cleaning

This notebook is the data processing engine of the **SnapVM** project. 
Its goal is to extract the raw results from the experiments (`.json` files), transform them into a unified tabular format, and save a consolidated `.csv` file for subsequent analysis.

---

In [ ]:
import json
import os
import pandas as pd
import numpy as np
from datetime import datetime

# Path configurations
RAW_DATA_DIR = "../experiment-results/results-json/"
PROCESSED_DATA_PATH = "data/processed_trials.csv"

print(f"Raw data directory: {os.path.abspath(RAW_DATA_DIR)}")
print(f"Processed data path: {os.path.abspath(PROCESSED_DATA_PATH)}")

Raw data directory: /Users/mariana/Documents/graduation/plmg-cc-ti5-2026-1-snapvm/Codigo/experiments/experiment-results/results-json
Processed data path: /Users/mariana/Documents/graduation/plmg-cc-ti5-2026-1-snapvm/Codigo/experiments/analysis/data/processed_trials.csv


## 1. Extraction Functions

SnapVM has different experiment versions (v4, v4.1, v5, etc). Below we define the logic to read each individual `trial` in a standardized way.

In [7]:
def extract_trials(results_dir):
    all_trials = []
    
    if not os.path.exists(results_dir):
        print(f"ERROR: Directory {results_dir} not found.")
        return pd.DataFrame()

    files = [f for f in os.listdir(results_dir) if f.endswith('.json')]
    print(f"Processing {len(files)} JSON files...")
    
    for file in files:
        file_path = os.path.join(results_dir, file)
        with open(file_path, 'r') as f:
            try:
                data = json.load(f)
                experiment_name = data.get('experiment', 'unknown')
                
                # Simplified version identification for grouping
                exp_version = "v4" if "v4" in experiment_name else "v5" if "v5" in experiment_name else "other"
                
                if 'results' in data and isinstance(data['results'], list):
                    for result in data['results']:
                        trial = {
                            'experiment_full': experiment_name,
                            'version': exp_version,
                            'baseline': result.get('baseline'),
                            'scenario_id': result.get('scenario_id', 'N/A'),
                            'tokens': result.get('token_consumption', 0),
                            'latency_s': result.get('recovery_latency_s') or result.get('task_latency_s', 0),
                            'tool_calls': result.get('tool_calls_total', 0),
                            'context_pollution': result.get('context_pollution', 0),
                            'success': 1 if result.get('recovery_success') or result.get('task_completed') else 0,
                            'strategy': result.get('recovery_strategy', 'unknown'),
                            'file': file
                        }
                        all_trials.append(trial)
            except Exception as e:
                print(f"Error in file {file}: {e}")
                
    return pd.DataFrame(all_trials)

## 2. ETL Execution

Now we load the data and apply cleaning transformations.

In [8]:
df = extract_trials(RAW_DATA_DIR)

if not df.empty:
    # Normalize baseline names for consistent visualization
    baseline_map = {
        'standard': 'Standard/Manual',
        'snapvm': 'SnapVM',
        'checkpoint': 'Checkpoint'
    }
    df['baseline'] = df['baseline'].replace(baseline_map)
    
    # Ensure numerical metrics do not have problematic NaNs
    df['tokens'] = pd.to_numeric(df['tokens'], errors='coerce').fillna(0)
    df['latency_s'] = pd.to_numeric(df['latency_s'], errors='coerce').fillna(0)
    
    print(f"Consolidated dataset with {len(df)} records.")
    display(df.head())
else:
    print("No data extracted.")

Processing 13 JSON files...
Consolidated dataset with 168 records.


,experiment_full,version,baseline,scenario_id,tokens,latency_s,tool_calls,context_pollution,success,strategy,file
0,v5_agent_driven_checkpoints,v5,Standard/Manual,N/A,4478,6.606,8,328,0,failed,v5_run_1778185266754.json
1,v5_agent_driven_checkpoints,v5,Checkpoint,N/A,8108,11.345,10,446,0,failed,v5_run_1778185266754.json
2,v4_1_complex_stateful_failures,v4,Standard/Manual,F1,1760,3.641,3,292,1,manual_repair,v4_1_run_1778176413143.json
3,v4_1_complex_stateful_failures,v4,SnapVM,F1,833,1.597,2,244,1,snapshot_restore,v4_1_run_1778176413143.json
4,v4_1_complex_stateful_failures,v4,Standard/Manual,F1,1726,3.052,3,294,1,manual_repair,v4_1_run_1778176413143.json


## 3. Export

Saving the "Single Source of Truth" for the upcoming notebooks.

In [9]:
os.makedirs("data", exist_ok=True)
df.to_csv(PROCESSED_DATA_PATH, index=False)
print(f"Success! Dataset saved to: {PROCESSED_DATA_PATH}")

Success! Dataset saved to: data/processed_trials.csv
